# Attention scores from named-axis layouts

Different codebases store attention inputs in different axis orders: `(batch, seq, head, dim)` from a fused QKV projection, `(batch, head, seq, dim)` after a transpose, sometimes `(seq, batch, head, dim)` in older code. Hard-coding `transpose(1, 2)` everywhere is how silent shape bugs happen. Naming the axes and letting `einsum` line them up is safer.

Write `attention_scores(q, k, q_layout, k_layout)`.

## Inputs
- `q`, `k`: `float64` tensors with 4 axes each.
- `q_layout`, `k_layout`: strings naming the axes of `q` and `k`, in order, separated by single spaces. Each is a permutation of the four names `batch`, `head`, `seq`, `dim`, for example `"batch seq head dim"`.
- `q` and `k` share `batch`, `head` and `dim` sizes. Their `seq` sizes can differ: $T_q$ for queries and $T_k$ for keys.

## Output
A `float64` tensor of shape `(batch, head, T_q, T_k)`:

$$S_{b,h,i,j} = \frac{1}{\sqrt{d}} \sum_{c} q_{b,h,i,c}\; k_{b,h,j,c}$$

where $d$ is the size of `dim` (indices written in logical order, whatever the stored layout). Use tensor ops only, no Python loops over the data.

This problem runs in **Colab** (PyTorch isn't available in the browser). The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import math
import torch

def attention_scores(q, k, q_layout, k_layout):
    """q, k: 4-D tensors described by layout strings. Return (batch, head, T_q, T_k) scores."""
    return q @ k.transpose(-1, -2) / math.sqrt(q.shape[-1])   # TODO: only right for "batch head seq dim"


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [{"__nd__": [[[[0.0, 1.0, 2.0, 3.0], [4.0, 5.0, 6.0, 7.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0], [1.0, 1.0, 1.0, 1.0]]]], "dtype": "float64"}, "batch head seq dim", "batch head seq dim"], "e": {"__nd__": [[[[3.0, 3.0, 3.0], [11.0, 11.0, 11.0]]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[[1.0, 0.0]], [[0.0, 1.0]]]], "dtype": "float64"}, {"__nd__": [[[[1.0, 2.0]], [[3.0, 4.0]]]], "dtype": "float64"}, "batch seq head dim", "batch seq head dim"], "e": {"__nd__": [[[[0.7071067811865475, 2.1213203435596424], [1.414213562373095, 2.82842712474619]]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[[-0.8019314252534474, -1.324358995628145, -0.24836162209524854, 0.4204452380655215], [1.1360465324896427, 0.10970639932180819, -0.5526473205362324, -0.7847803553442784], [0.7487457707345911, 1.6347830429585775, 0.27276877584472176, -1.2333286640307717]], [[-0.9582652054360887, 1.6000190889991115, 0.2028824405086084, -1.7321348424395848], [-0.08369619281702581, -1.1632259734447485, -0.6292880940615545, -0.48800582327685743], [-0.7133133716322436, 0.5533784703532895, -0.06308597192528916, -0.5894312580326048]], [[0.40963782655711695, 0.8298553070613239, -1.643023371405677, -0.256730126365494], [-0.9807473560440125, -0.17315522486203205, -1.2894187467538587, 0.0206903940375912], [-0.03788574104406823, -0.304337750958489, -1.0479265051202462, -0.3961903304730927]], [[-1.091328901695709, -1.3552087462047395, 0.22478573245989314, -1.109349937891366], [1.1702961011782933, 0.7165876558738361, -1.9978166924497212, 0.272128869412488], [-1.1017166275810448, 0.033057220158269195, 0.04363199256942161, -1.9884297882311208]], [[-0.23342252376577002, -0.255790031399391, 0.9620005318430944, -1.1814468079562157], [0.7380418978456841, -1.0989727630364063, -0.33129089269991674, -0.8404731684222111], [1.448731288921672, 0.5682130997882933, 2.4317325028452124, 0.6419163790823205]]], [[[0.8449927337191754, 0.8406828762653401, -0.6066115359095516, -0.07002844663838208], [1.350388867744626, -0.3965507651729716, 0.18879953129109867, -0.021223460417289796], [0.6092164928327407, -0.3649087419473726, -0.15236188875684828, 0.24238142867215043]], [[0.10302313848680769, -0.8649727462818201, 0.8957830431894438, -1.298481208246912], [-1.2011155488035266, -1.282491794740462, 0.9669722789332148, -0.36060836889927], [-0.9710363785210655, -1.1360213941896466, 0.42113113746240616, -1.054840662577835]], [[-1.2720782100976422, 0.6139930624688609, -1.1967077271925706, -0.32243815075139726], [-0.006761540346521785, -0.4453353669298123, -0.05409396159544828, 1.3387734990547715], [-0.516894152589275, -1.2593071588782276, -1.8367457051936795, -0.20476611683386706]], [[-0.3522570104112305, 0.2650909069303235, -0.4642445916374816, -0.4786384630527245], [-0.7213157478486046, -0.5197597723701788, 0.1602267063174243, -0.3803527272151388], [0.1004418528325065, 1.9011872000332568, 0.4791181507969435, -1.5760898454352261]], [[1.7335295856131068, 0.34781036998853543, -0.9414132864509847, 0.9070489576858102], [0.01765561090983897, -0.6152185330350691, -0.6335089254336164, -0.9934317902518573], [0.048118983989838576, 1.0688166937495658, -0.3250520988063787, 0.42082412505635486]]]], "dtype": "float64"}, {"__nd__": [[[[1.875646053775279, -1.2145906329798497, 0.2575223239234917, -0.30643692685187435], [-1.0592560998278446, -1.0258471994365972, -0.0152851431653622, 0.4338634112001693], [-0.531589820743032, 0.05439054435949247, 0.7222086544347099, 0.2965612469178253]], [[0.8686701897408925, 0.4613956081509272, 0.4880056756807095, 1.8266340649418156], [0.6231718736925668, 0.12947113235560923, 0.7099438034419987, -0.9191236755082103], [-0.3377047857608698, 0.793220854194603, 0.6307867000938839, 1.5483616908312823]], [[0.010473028918682835, -1.4623277904204515, 1.9472469308762064, 1.0928928105587499], [-1.0587374336035666, 1.3758236699684163, 0.03240279877908362, -1.813928769904867], [-0.4203539214229468, -0.5089038339937678, 1.590848160322242, -0.7920686954195238]], [[-0.25361734037936806, -0.2767608068453714, -0.3789727629772628, -0.9140740610326185], [0.21908690902567676, 1.0768046464135546, 0.6239776511717521, -0.9274991160593102], [-1.1498157571616898, 0.11897355358204866, -0.7065801636237695, -0.6301656488493008]], [[-1.6796945788401643, 1.9504916026999646, 0.9166191525408893, -0.9739073319069764], [0.9082102667945468, 1.3424871058246228, -2.389553349676638, -0.548944785564265], [-0.3879732181256381, 0.6482544703432906, -0.12146542230081914, -0.2304313358523204]], [[-0.05837138907215449, 1.8533257078420022, 2.159980469779012, -0.5248264370136562], [-0.9262439932504336, 2.6925531473868567, -0.9797446331927372, -0.5734020178922011], [0.036581509348738155, 0.4832462441452338, 1.029035549437811, 0.39097722167013954]], [[-0.8723830716951919, 0.5063823263497103, 0.24991757985122218, 1.876918992454176], [-0.01499979815571943, -1.3369728999172332, -1.0450199143839616, 1.450153058953162], [-0.5401312070783946, -2.1044660100855603, -0.5806997054894565, 1.5099831293121058e-05]]], [[[1.1888306785373703, -1.014468137602427, 0.6666833259020761, 0.7952990996016167], [-0.6993883083236738, -0.18758970531896946, 1.7694502363979239, 1.720484746826155], [0.8555220049018919, 0.33194635951240653, 1.1383096209632015, -0.14067728561527593]], [[-0.09509306569845781, -0.8603601395273159, 0.005563384668739004, -0.0820166249133826], [2.7736316673693673, -0.19284316578043667, 1.2708179900134384, 1.3204972743933365], [-0.18647870191497867, 1.1693971923001296, -2.1768136494850046, 0.09492718485511631]], [[0.8572613994662929, -2.39786527267812, -1.159171198248098, 1.0560736173734917], [-0.25701076898467345, -1.0981986938209587, -0.3735956685207676, -0.5425883064344673], [0.7244473133352144, 0.4538858794596457, -0.2802283393557666, -0.6653426856286421]], [[-0.0538756638943387, 1.325155927345223, 0.3490261125780868, 0.6389455936428303], [-0.1506815972085555, -1.2918162500762125, -0.7108435355871252, 0.6512365658972803], [0.10321610258141592, -0.9038398688268016, -1.617870425602848, 0.11068190035142705]], [[-0.339531008698164, -0.350530218612727, -0.3294499574610804, 0.28390511348310316], [0.29444772495280114, 0.9621285502204014, 0.9284215804413863, 1.3320930804934341], [0.8063630567820633, -0.3356080333276904, -0.05426894624359061, 0.4942150948314312]], [[-0.31029227000142345, 0.5494387802798892, 1.1536094769949883, -0.8012460099660716], [-1.770384050846499, 1.1936549552915572, 0.01775991033501458, 0.7086698231113021], [-1.3323053543858547, 1.7329925604781644, 0.7386198894684642, -0.12607417369146828]], [[-0.3519880388050872, 0.11389588045524304, -0.16673134056534972, -0.2786901272438552], [-1.1195999755255746, 1.262809438921045, -0.039577684569559124, 0.07463563738960281], [0.09291598860047003, -2.358217336741871, 0.5271414581243252, -0.32915493062535894]]]], "dtype": "float64"}, "batch seq head dim", "batch seq head dim"], "e": {"__nd__": [[[[-0.04419204557007333, -0.3304348172213958, 0.9520642421679676, 0.13985929168248015, -0.9366394759595432, -1.5823878755992284, 0.37801556400664316], [-1.578848856780171, -1.57957168150752, -1.9238819124331936, 0.6533113546935639, 3.301660710751222, 2.184292024382643, -0.777088581415992], [-0.2920212578688136, -0.2660124811194008, -2.3445905277131107, 0.26188438772837247, -0.16271798329536558, -0.9500392953595148, -0.4148095596111662], [-0.0015448844925796518, -1.7449859604866529, 0.5978213241854055, 0.7903444661485904, 0.23811145314280094, -0.6900959288019091, -0.880089479783146], [0.24131884111435856, -1.0046984254440898, 0.4768310456229951, 0.4226752686198006, 0.9627850301091175, 1.1187599176948282, -0.9514764002825153]], [[-0.8239732051653377, 0.5255588609837577, 0.1768950076818257, 0.3750343963657433, 1.4652149453680492, 0.1172909598555934, -0.36211956442058035], [0.5399193486907756, -0.10049138713262931, -0.3234822140823991, -0.6054703051687653, 0.0669880564856666, -1.0790797516835542, 0.7531962408706452], [0.6225895884263204, -0.7840123254182747, 0.36040560230456786, -0.6125410303050447, 0.9732967634745551, 0.8468094425606632, 0.8118436810221894], [-0.9130745249876353, -0.4231923192497743, -0.40574763489564974, -0.23548539010065037, 3.324695234050561, 1.3233907769217534, 0.7533871298440035], [-0.006991989076569624, 0.42747128780149823, -0.3897808082262378, -0.22443182584370394, 0.22397554524982347, -1.4180699808734494, 0.29280858329930837]], [[-0.2389352031602809, -0.34684548396122944, 0.1320645760007663, -0.04097769289838817, 0.5101638532292784, 0.3079391842829805, -2.001590837783396], [0.09446248992328506, -0.13630294453758635, 0.19236874648326446, 0.6510154925630633, 0.3894813739625963, -0.027023957898482137, -0.371329132060889], [-0.4353648556423448, -0.7515384434021586, -0.5912389632610519, 0.49873154821373045, 0.01799592395410871, -0.6908555071601432, 0.6347281770049943], [0.01463976500117193, -1.3265046871478772, 1.0453364110143197, 1.2464573602582525, 0.4508813950786181, -0.3784297811350784, 0.25006831275668673], [0.6036801971248027, 1.24464912094862, 1.2309648031024292, -1.8604496667513157, -0.31850566121750523, 1.5404487490627896, -1.6951933951048221]]], [[[-0.15420203183069398, -0.4006371513708514, -0.3311265974469578, 0.40602185917642686, -0.20080963850633468, -0.2219868712848163, -0.040509813740764615], [0.2822435206907644, 0.42293595923231214, -0.12362589011512226, -0.8343906874347436, -0.19777080226879754, 0.7832849204411717, 0.03886952026416423], [-1.5947109294569166, -0.19375089832130202, -0.7580527414694641, 0.12923396130743042, 0.2597001727200406, -0.19505672937916843, 0.40353830385158357], [-0.6889325432205665, -0.07895145529528304, -0.45248355553473746, -0.048796277559383754, 0.02186862915176776, 0.051452025066061884, 0.18248947689069903], [0.9008906468652594, -0.2718596798063458, 1.3506276160431268, 0.3092425155546906, -0.071420344699045, -1.0797964859939064, -0.33319500044355266]], [[-0.28625030462809486, 2.0169288755123884, 0.01470398267733045, 0.08038177558370925, 0.08155002579310447, -1.437869531480916, -1.0108598336704302], [1.085608279453941, -1.165734472444152, 0.7757678369177804, 0.4577612004967373, -0.5850968467111959, 0.17860209844139288, -0.16997928199858994], [1.1479460419805714, 0.8831145182763466, -0.10769453442385424, 0.7433104511928631, 0.6513440610019009, 0.214090715269963, -0.2263711786332244], [0.1155517504869446, -1.0995339195914016, 0.451350178399394, 0.2092637033729059, -0.5351864006710041, 0.19494658989336358, 0.05824913842308389], [-1.363543112268532, -0.9746438297574158, 0.7233975851994443, 0.2977277697577948, -1.2491138161521027, -0.7404410276451517, -0.42287179525205915]], [[0.09626771080734335, -0.09282856662772192, 0.07857386728954435, 0.33301452620900224, 0.3708865292693991, -0.7935710451001319, 0.37852138187545337], [-0.29003570444288684, -1.0821198440999762, -0.31763556255467246, 0.0642337278750529, -0.4729612388285334, -0.11547782957637612, 1.5789811142927472], [-1.4611080573127087, 1.3012873727202237, -0.14754824027700003, 2.0169064351314536, 0.002154499060464761, -1.412275140659712, 1.0104336054508196], [0.7420636135429189, 0.5059740024724958, 0.925032110768154, -1.328798639334542, -0.6809948636533097, 1.8567572379557473, -1.8513695885494355], [-0.016626833443424856, 0.9942117850342739, 0.1655385150029153, -0.19430134748847785, -0.047142113904616284, 0.7474985998189111, -1.412947903525528]]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[[-0.2892284645923461, 0.06978787257481246, -0.9724756053284209, 0.8433808099877073], [-0.6958646238070096, -0.9036580547128793, -0.8621896009823745, 0.15875269513992676], [0.7457794743624686, -0.5950532355462611, -1.510186675915287, 1.062490693153603], [-1.0650835950433684, 1.194830368529916, -0.05429587697711575, -0.6983498940240875], [-0.6400399491357172, 0.4618661736925272, 0.8893717949967992, -0.19817022758122477]], [[0.946934382371629, 2.08697820766597, -0.20830834526001082, -1.1494466412305686], [-1.0544003485304514, -1.3903353550011621, -0.639665450940511, -0.7237792369318543], [-1.3251033289662522, 0.5301842547135, -2.6113390049731797, -1.3207989716931114], [2.0152518061838767, 1.0378024284634166, 0.5227916833432323, 0.45725905895534386], [1.7709371728004815, 0.5356296362853685, 0.0775770872272132, -1.2301601607845671]], [[-0.07366681732892769, -0.7338839983148214, -0.7061752749030227, -0.7142389333341079], [-0.04464686469247794, -0.32971081974207533, 0.8711105109344268, -1.3261814419117322], [-0.16947019674286637, 1.8594974964345012, 0.0388973371949769, 1.198615221448642], [1.1368634972033143, 1.3166478203373606, 0.7381191695252466, -1.3804460312528912], [-0.2673877057870027, -0.38033558246862037, -1.5488553043334206, -0.4638816211751379]]], [[[-1.2500798351337994, -1.6655514460193832, 0.21089625656145172, -0.04359127244566091], [-0.6878149095751852, -0.9505863626128569, -0.541273442072544, -0.39409036937385966], [-0.4081102825826052, -0.052707747928468524, -0.43554067396098334, 0.3869719823711992], [0.5228708948391112, 0.35786967764803124, -0.6631035145259176, -0.6165061466820784], [0.4348641034709542, -0.197042778476594, -0.04315170860457781, -2.4919952503112044]], [[-2.700103979886236, 0.33714819206874613, 0.3295586563507561, 1.3934318714435818], [0.7211631188958838, 0.22173880016804176, 0.01636245608868217, -1.426740428209398], [0.8831758896325703, -0.6035860153728927, 0.5024844545833684, 0.20004164543806374], [0.272097020758701, 0.8926752155435728, -0.48986611197868063, -0.3872351816195507], [-0.8558614733519333, -0.37715202345150317, 1.6659407140512403, -0.3633186666203786]], [[1.1856163121936107, 0.8260114337421408, 0.7292817915893176, 0.8030151370721609], [-1.0142765548969686, 0.37344767888543307, 1.588412097484749, -0.6957959363309963], [-0.0471070855877405, -0.6891591571615298, 0.31701151121305016, -1.0785301318658582], [1.6096797942508279, -0.37683478895943645, -1.542661572000243, 1.790790910814277], [0.970522894800899, 0.518511481410699, 0.5325875434742989, -0.3577980078127549]]]], "dtype": "float64"}, {"__nd__": [[[[-0.22329559977220623, 0.7240192298296324, 0.9176249577098493, 0.9326124031440995], [0.2125509856107767, -0.466388093105908, -0.19208732917753973, 1.1095997115146592], [1.4399181149572342, -0.34131239008610353, -0.47730400254445376, 1.2475496699476993]], [[0.7236490231334424, 0.6972581900253083, -1.3123336629533346, -0.3737574414663936], [-0.5149955292041763, 0.20790465904469757, -1.6849419797309495, 1.2717528081338776], [0.30013179535055773, 0.43483702547979786, -1.1841403768848728, -0.026420371368168806]], [[1.3346502881568998, -1.5696956024589206, 0.3057680573775562, 0.6988049143143882], [-0.6919471030872025, 0.8147412282685731, -0.33006903056415593, 1.2297665530315567], [0.5275117754251525, 0.22436588572838914, 2.008915981414082, 0.8318964583740944]], [[1.3086357086483342, -1.1895196722305454, -0.6466070677186754, 0.8365690029317673], [0.9056118633317605, -0.05574002999013012, 0.7797265342256705, -1.0583537461465884], [0.21383522572037922, -1.1472838649378465, 0.3405705337781045, -0.4861719071399926]], [[-0.6409975166442754, 0.05629903018263689, 1.8174600636946423, -1.011761734455417], [-0.6017506161418752, 1.067664145735261, -0.4235666699986711, 1.7692642772096374], [0.22252970384637974, 0.09063191415105347, 0.35688413886287884, 1.741179596515403]], [[0.6553615344906699, 0.5167121329088065, 0.2518577631500006, -1.562418833300465], [-0.8852720281451512, 1.4833441210372633, 0.8218783319217637, 0.5060648618402573], [0.6370627277466291, -0.5435930521122606, 0.33796426767211185, -0.900794630349013]], [[0.39136232136016047, -0.7141351107069186, -0.21381982915618122, 0.4534148448314499], [-0.32678770337045643, 0.6475819228802786, 0.45603827622850734, 0.35945813704442325], [1.3244113828426742, -1.4738846764712352, -1.5962822788262483, 0.6162417919107772]]], [[[0.9407940950142847, 2.0116404277145703, 0.7562567989915012, 0.8302149983209312], [-0.8830800515074957, 0.7526217774690592, 0.6738389380752741, -1.0558651891629185], [-0.896317509048086, -0.11103120279927593, 0.2882537511901322, -2.944785538646996]], [[0.0703055101868928, 0.23085449610631834, -0.17683635976985834, 0.26682489806649157], [0.4559062316030521, -1.011638037208477, 0.4161801684774495, -0.25130719973217774], [-0.9458787787894589, 0.20914079384282933, 1.487644239755298, -1.4601155833251631]], [[-1.653109915764168, -1.653166149925423, 1.9773117527941952, -0.25158374870191025], [-0.05948675368658991, -0.7865757190025763, -0.9404477838814673, -2.0268601600545053], [0.1704062939055249, 0.2595598209214102, -1.9765340524052937, -1.043149176525692]], [[-0.7441137561592083, -0.026002454439472283, 0.24412916699413742, 1.1257929683514525], [1.3644560191461277, 1.068454512203335, 1.341528531502859, -1.1664166902953155], [-0.5544917649796093, 1.1436098002700195, 0.2886610787245648, 0.018721718675669165]], [[-1.9789881677034367, -1.8592429271083921, 0.7308894225311937, 0.34157639490844327], [-0.29605762418597315, -0.41297173199761555, -1.7261026268818427, 0.3712151390391477], [1.034420799017549, 1.6421709791918642, 0.04913795013614401, -0.20419668212000805]], [[-1.0959970457174089, 1.2232436651983958, 1.706855529223571, -0.6131700061123667], [0.2907587696302524, 0.0906701056389038, -0.5738273234080787, -2.4974495618193604], [-0.3659991054773752, 0.6074441438068976, -1.4433020292385828, -0.8388812655122865]], [[-3.2835604878882627, 1.2845277165004347, 0.5002763718002052, 0.04547852895218472], [-0.2704733880742747, -0.637283176318838, -0.07880625474327216, -0.31031493210381433], [-0.24835672044910614, -0.03381111303746043, 0.1256509655342884, 1.54897841817735]]]], "dtype": "float64"}, "batch head seq dim", "batch seq head dim"], "e": {"__nd__": [[[[0.004645361490153494, 0.4001764447015709, -0.10177894616373712, 0.4364235494293121, -1.2157061448151265, -0.8580642808646373, 0.21365242673538307], [-0.5709971319442947, -0.030749649412937948, 0.16851963903404746, 0.4272956057731256, -0.6662213228093481, -0.6940806667826454, 0.3146664061168932], [-0.4961261231469622, 0.8547659276009834, 1.1050570545211202, 1.7745626693258074, -2.165617554923268, -0.929561876574088, 0.7607376521809218], [0.20089780932332776, 0.19731316689763248, -1.9008231924417334, -1.6820952151269755, 0.6789334212138677, 0.4984043587558706, -0.7875682897013785], [0.5543059196388933, -0.6171046299378169, -0.7228798235887997, -1.0639175950592679, 1.1265827039801417, 0.1764071895975603, -0.43017050979509147]], [[-1.00374110908482, -0.582299499400374, -0.14983810735681094, 0.8976621986152724, -0.14353447576769554, 0.7522568314284586, 0.2669341608433611], [-0.12795579138667632, 0.20564258089065848, -0.5410600051296328, -0.3050648941410074, -0.9297704666079745, -1.0104605267192295, -0.5538356200298062], [-0.7464383325378036, 1.7564374892076864, 0.29325605792127213, -0.9339197767741769, 0.06633709631517734, -0.4275451428932925, -0.44463863175079327], [0.175638961880615, -0.5607174360040004, -0.07957133567396564, 0.8454407570824332, 0.2414602065993456, 0.20822298825998667, 0.2081405451576951], [-0.6266418890794351, -1.2479186797270818, -1.1637054566422016, 1.4681796143994041, -1.3515637365268798, -0.6660099877043364, -0.31933469011172855]], [[-0.20478937178400564, 0.25692571250880264, -1.1081691407043, 0.46648056177454517, -0.7892736896019541, 0.3783622226255712, 0.835603202479065], [-1.0110075974993094, -0.5766246691557307, 0.27460728766968856, 0.6550757818564189, -1.0190260598360368, 0.819903502521869, -0.8904808952943742], [0.2990367140392265, 0.33999359951288494, 0.7015377592168087, -1.3695479667863415, 1.115851937558514, -1.0926666552733704, -1.1442937308655514], [-0.44344004762183115, 0.0380856447479715, 0.6147754838498082, -0.17247580913289579, -0.8839326149737986, 0.7507445152061659, -1.2319235880902943], [-0.047321840931851336, 0.8003412414792627, -1.8619030723874828, 0.03860385065909732, -0.7272176771693522, -0.03459531815026025, 1.1964932495233993]]], [[[-2.2016283753575947, -0.26065645175985125, 2.6239635724634645, 0.48796060332386154, 2.8549050208209374, -0.14029657339563517, 1.0343948523973836], [-1.6479258092430957, -0.13862030568966008, 0.8686956192252344, -0.019637343387207826, 1.2991627688831078, -0.5455934053104872, 0.37435918316638095], [-0.24904371639040201, 0.06970648242367096, -0.09838486791687281, 0.31718658044510684, 0.3597451754388183, -0.2989356875354343, 0.5360292219320278], [0.0992548854507707, 0.03606907552112965, -1.3060202716033265, -0.6271610545227996, -1.1976806364747923, -0.4445509811862606, -0.8084787518712918], [-1.0443936220333827, -0.33610522812954846, 0.07424370812534761, -1.5672350431120974, -0.6884932021105437, 0.3683609536004211, -0.9079647782026407]], [[0.6944731408601408, -0.8925449250929723, -1.6193982076052145, -2.253577735961043, 0.30428231999126665, -2.2118226136581263, 0.028536700428702544], [0.5237559489446225, 0.23491161341082514, 1.3295506440573583, 1.4535188649023691, -0.4314743104273846, 1.8918063072340698, 0.05254177811030881], [-0.5534052041887272, 0.5860540613921627, -0.2278939979123258, 0.500457887706522, -0.4026441216218003, -0.29293465105600713, 0.022052405184777968], [0.255170321957418, -0.44278552371836355, 0.26361112182521274, 0.5597777464101803, 0.12630279676638398, 0.7041262227961299, -0.24185626358453316], [0.9890663110074904, 0.387992418704826, -0.2413815385778286, 0.5439664181100693, -1.3006582966571039, -0.1658189494613626, 0.22664852547281455]], [[-1.6544454267554765, -0.5181398406916257, -0.9313395061369604, 0.25638469068433795, 1.2273702817937981, -0.8291934728291356, 0.5065519665452091], [1.6872425234472532, 2.2082102048768073, -1.2448193689888793, 0.7174863381535819, -0.10789664387792056, -0.5553976553015, -0.31945582899932784], [1.6930803841717648, 0.973402650295272, 0.15578920035303295, -0.34534586398116085, -0.47231807336200715, 0.022915410752534924, -0.797893247847353], [-3.559558462762731, -3.2555334008868075, 0.6787747311117883, -0.8676429088720824, 0.30239115739571587, -0.04689090938618713, 1.0965129826388142], [0.13984556970948295, 0.2525860652112186, -0.18973638071864463, 0.10093327998605162, 0.9773225008370436, -0.2543891925573912, -0.372934293341225]]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[[-1.4494324054647152, 0.26196862488829153, -0.6245608174151475, -1.1394811867949461, -2.3681708051255725, 0.6068921428382499, -0.9903792389515337, -0.7596859730023938], [0.45987869267679876, -1.0934730110875337, 0.016045130164280344, -0.08488040408826328, 0.7918433900417065, 1.0006409423733347, -0.8768518458101385, 0.9071777346550384], [-0.10674565894923128, 0.11279756399596276, -0.2225136131382745, 0.765539634412858, -1.2222004081820923, 0.250084266960452, 1.1657797884877044, -0.29366991481397603]], [[2.017772471985153, -0.7459058475973274, 0.2036961304237924, -0.49260852892014195, -0.6287503105652976, -1.1257709935845257, -0.5500664691168721, -0.06223707311992144], [-0.813097176129281, 0.8723957394787941, 1.5103266276144978, -1.5388961513985318, 0.8248433607793892, 0.1780600818362494, 0.23332070694060292, 2.4504993275728366], [1.1943246457946253, 0.3243098464303824, 0.17311415244634476, -2.1657171481499815, 0.7723909164870639, -0.7486869461197863, 0.1757326242531968, 0.0976306297693364]]], [[[0.9603515134967532, 2.0305028814674015, 0.9836022675228114, 1.2729486704011452, -0.2345432028281203, 0.4557735137751234, 0.7692489297752649, -1.4396281852276858], [2.0415429191664223, -0.714868246010375, -1.2775433055562428, 1.6340442221712168, 2.11843861535627, -0.7031057381505373, -0.6133099552415482, 3.1119129382202093], [0.11261711124014562, -1.9840581343909514, -0.6122687481297199, 2.0021187846260236, 0.659124064155549, 0.7520990806911315, 1.2245574277845088, 0.6757786896026271]], [[-0.3332044808876784, -1.5943117200745838, 0.4463708390586508, -1.116034915943622, -0.05623851864069769, 1.7589710766812527, 0.5406502875557727, 0.059491529994791854], [0.0656800558330705, 0.7307927491415821, 0.9212337664614076, -1.3168137745000827, 0.18547932372329412, 1.715647072880231, 0.7543686438915898, -0.7651259000530994], [-0.6796276894565825, -2.061922492127639, -0.793378841071178, 1.6069652687673104, -0.06952390178003978, -0.8550109597366835, -0.5761205222847363, -0.11074551402067748]]], [[[-0.10654617191788289, -0.3539737132397008, -0.7690944405035621, 0.636570086422678, 1.538534566643514, -0.5679570038398728, -0.5412718676392838, 1.1893530639173242], [-1.4637436480554302, -0.591394697898987, -0.37809709746053366, 0.16778159182906616, -0.27493880956242583, -0.8866033394203041, 0.3105063468104034, -1.1877104022986131], [-1.7425136392104728, -2.0282575532960294, 0.7486985378512533, -1.7368555460158381, 0.44705728629242797, 0.670709123171108, 0.8068993146395768, -0.17468272255542663]], [[-0.5482001218387921, 0.0856713362219482, -0.5937349097040746, -1.286202262300972, 1.2554399352377776, -0.8944374586353495, 1.795889000036988, 0.5147645516042764], [-0.4545567548775306, 0.4376476468228372, 1.2301494746406227, -1.378469250276322, -0.926772947442875, -1.0492675511760912, 0.16503139897918395, 0.4042508713136647], [0.028968342989254353, -0.7282098404051072, -0.5284513163621615, 1.1870432673580795, 1.116942674442043, 0.13135528984481942, -1.3106089423110556, 0.6281267915471365]]], [[[0.28765917679832087, 0.2527304950821622, 1.7300316088755747, 0.8205397313385244, -0.9744854542313894, -0.9647605552913328, 1.3803138486051605, 0.2057165176789978], [0.8870580070430005, -0.10639959608685703, -0.9494908518002053, 0.3242014124060831, 1.618915491436607, 0.3991358441971388, -1.0655647560808246, -0.24585874835996752], [-0.4284127077441873, 0.24734402051603865, 1.0980541380077837, 2.054289901567956, 0.31449121970026706, -0.3364865964606304, 0.17159816274806688, 0.11086616939243021]], [[-0.3297963407584658, 0.11860415855321144, 0.09529158095838645, -1.0805187404242564, 0.6438245074939835, -0.26670256295731154, -0.8877440710825989, 0.5859832203034813], [-0.39159296460502474, 1.838255221034548, 0.5709067180842602, 0.1686367445319976, -0.6657762623743938, -0.31839120057526804, 0.9355282866027194, -0.6024860526591809], [0.8436412707458482, -2.0179640442538918, 0.2804525786513644, 0.8953021558456109, 0.41629401487851075, 0.07274018953469119, 1.0863400275074944, -0.7792912035706152]]], [[[-0.9079190420445703, 0.18021408052490404, 1.2776237015139034, 0.5065013799515, -0.11938237415906136, 0.3750716036436929, 0.5171626722510868, -1.1704579225994451], [-2.737101308424054, -0.9506103349906968, -1.4196819925965811, -0.5050193086315621, -1.1275153636412767, -0.7912658410106157, 0.5087776489195652, 0.1735101418875656], [0.05337089932598016, 1.7401035051472364, -0.3785295647557631, -0.2968817458113318, 1.3511509205997787, -1.0253494858067438, -1.5425773917429322, 1.3973295584318948]], [[0.4284773896230237, 0.6939471341402738, 0.7324030743505717, 1.0005126796942065, -0.373421250381592, 0.024766209961962146, -1.0105914896329433, 0.24289265325355808], [0.33745420956589356, -0.34638737757412497, 0.7765704945224762, 3.0646240121743187, -1.0395714163213274, 0.9449557567607936, 1.060904820764859, 1.186570179751149], [-0.6814989053316384, 1.3214032858486306, 0.7814787108169615, 0.0047557062710890646, -0.21026098035303756, 0.38018947691790894, 1.0715250146520645, 1.0241744207011083]]], [[[2.4328604881410856, -0.20646864420110606, 0.7699827432898934, -0.2774447580275897, 0.9362072176514797, 0.28631535526570023, -0.44236699240730226, -1.1707319757858579], [-0.6131279592631117, 0.39718727745368554, 1.6269680845754784, 0.9308051712316341, -1.0413693363453491, -0.33361267495752045, 0.18406483453498276, 0.011326163811934926], [0.19017614050865247, 0.891305115149018, 0.3138678698529088, -0.6011586258767794, -1.8446870347400581, -0.2656921423057748, -1.6127432706632423, -0.10446329880547667]], [[0.3634338444161454, -1.2998757087597277, -0.11584888412212659, -0.32716659712257967, 1.9060788306639675, 0.448768092469958, -1.9007844918971633, 0.4204667127804782], [0.38629809819080335, -0.23493279640908496, -0.41376384324268234, 0.5544675350795898, 0.6138501359810715, 0.3820300691786012, -1.0521268312007939, -0.3142268610437008], [-1.6254592103256966, -1.5668897740611265, -0.6347933471206841, -1.1325814508998646, -0.4956102868976653, 2.392772217170665, 0.4025426374596253, -0.980901437513614]]]], "dtype": "float64"}, {"__nd__": [[[[-2.261459792935038, -0.9428343485501535, -1.5433629844530297, -2.311529977901388, 0.605044677013374, -0.602545854033424, 0.31645127035302306, 2.966238210900678], [0.14657582474666686, -0.00741881958223305, 0.22416597109845818, -0.39149740020283813, -0.9520021536448358, -0.33749387102527995, 1.0182347009753103, -0.39336620261872396], [-1.4920489584580787, -1.368557102107097, -1.1497435906932556, -0.1072094578765301, 0.365708100070376, 0.3683388201245052, 1.0810695463216409, -0.3066530564769316]], [[0.7394214562816036, 0.2368479351225408, 0.6244451117724273, -1.9157264678312913, 1.179128564389489, -1.6059455496125163, -1.1812087238869644, 0.7285397875893229], [-0.4316533455947616, 1.7289521681788165, 0.1270528148073479, -1.0572758699119627, 0.12839329190039186, -0.4259935193114223, 1.5300532577009178, 1.6091804130260385], [0.7201277077487529, -0.42249883369204827, 0.06931524067415851, 1.0352593349259305, -0.8136550558539355, -0.806729876970593, 0.8386394417799221, 0.7259921165979379]]], [[[-0.24201310532798542, -0.27826632513265287, -1.9611249766590104, 1.1736487550830053, -0.1261032036505945, -1.4362707237074057, -1.1350211878512138, -0.900301092920432], [1.6969619837176821, 0.6850574772130213, -1.3414423058366864, 1.85128431564108, 1.3351791816522625, -1.0612408977110352, -2.660992127169175, 0.9614986162699597], [-0.6027781457301238, -0.13628479100503227, -0.5249081791935718, 1.259288122334379, 1.4789766959054405, 0.8101177912610477, -0.1506368553091435, 0.2618435842983696]], [[0.6211815737480053, -1.5746363883228522, 1.2433815123598415, -0.3186997372746957, 0.26307061762357126, 0.14000190648152444, 0.9083163692171554, 1.0714817689176728], [2.213199810675672, -0.3474785581572002, -0.8965520829041624, 0.1448122370416971, -0.6418266810564457, 0.8751027737365411, -0.7909221451886773, 0.15036309924915556], [0.9588245333478886, 1.0343275199445572, 0.2264312628499982, 1.4341362820283172, 1.310883093403755, -1.6749932273685297, 0.38531661986118254, 0.20764310239580833]]], [[[-0.26980638055382183, -0.60727273635687, 0.07507789217605107, 0.4187395477744383, 0.30865448854044664, 1.399886622511851, 1.028114000085676, -0.5532593446053599], [-0.7224798425179273, 0.8842297105940803, -0.6599516844403207, 0.20378567909571624, 0.5136725133708302, 1.067847221996037, 0.554870562997444, 1.4836307120685868], [-0.41991152579493224, 0.7100406584423806, -0.6179177338304227, 0.22685877595638926, 0.16443013069289658, -0.8879100723553962, -2.563278464806261, 0.9970925298778534]], [[0.4042292919261386, 0.18224069514569657, -0.6591445939728919, -0.2914599071297303, 0.4969172707031677, 0.06155714267584536, 1.2008442105159, 0.15820167857532091], [-0.7803479255335084, 0.6506231924283851, -0.5935804333439826, 0.2554992282540186, -1.7722288885231972, -0.312556799006476, 0.593013104847937, -0.6167822577566933], [0.0055620633214661895, 0.621496796021381, 0.09812398632467413, 1.8157239339574112, -0.22971957825847528, -0.15062332329935044, -0.13217906594333403, -1.9484788261880894]]], [[[-1.5057890875226683, 0.5606258593966804, -0.30178528979364067, 1.4161562435228485, -0.050887887787709446, -0.14292546560026553, 1.055790401389666, 0.3035111621653347], [1.630901467114023, -1.0298982323665045, 0.4672877277604976, 0.7073956571956805, 0.5844881894017102, 1.1814675210823202, 1.0607036949765016, 0.0700731922747085], [-0.5216994302852448, 1.0194790729530288, 0.09296745528517156, 1.006192428582429, 1.404382125417232, 0.2584978608813258, -0.8391388200385301, -0.18870086819908075]], [[0.8577681454614862, -0.16371336762469718, 1.255357696153767, 0.12203248991171817, 0.21839631310269905, -0.5367452738192108, -0.008135730447932076, 1.1862749094883955], [-0.2507839039830373, -0.243149257385674, -0.690904744947125, 0.8662309713247346, -1.438070302821455, -0.5731570593110245, 0.987477967649231, 1.4973156156913698], [-0.14964297894986278, 1.0339836199005135, -1.9149897619140834, 0.8238314140215098, 0.3572783410259209, -1.1592878469988148, -1.7167603012980874, -0.44351816493908386]]]], "dtype": "float64"}, "seq batch head dim", "seq batch head dim"], "e": {"__nd__": [[[[0.8002197495211014, 0.49511467375717927, -0.2727107890852629, -0.11958216662705157], [-4.5927018287778205, -0.5071592889008479, 0.44820122482895897, 0.5374503983579692], [1.739465051649839, 0.8997897001395054, -0.38261165760192895, 0.3139442278909754], [-1.5615993947726579, -0.9945724277842891, -0.036641692243548234, 0.7267945093749968], [-1.7203022718200505, -0.6358375411637202, 0.7463566832528765, 0.686996836311701], [-3.2077615896968616, -0.47381946974191896, 0.10369770014283082, -1.8792569874112568]], [[-0.788033111602099, 1.0795827493032861, 0.35625329218980994, 0.9200071397638254], [-1.5024748379744084, 5.625847125704893, 1.3021464946665098, 1.6262606230547345], [0.3478026530611075, -1.2253027437110195, -0.6574340858047103, -0.9893132530651514], [-1.0158141708907713, 2.7023215684280233, 0.09175306737462835, 0.5700216331899751], [0.4509839851619593, -2.184686780608838, 0.3841550188610336, -1.961377257689693], [0.42229500071794124, -0.9697735298258788, -0.3047956736478379, -0.2818148321736761]], [[0.4151251774180077, -0.2572835068848475, -1.1554033362077825, -0.5849003100248744], [1.651535938577618, 1.6340339349269297, -1.2897647227799054, -0.05617184793841606], [2.134585624648509, -0.076430242968392, -1.5307451017989147, -0.9473907451294508], [-0.3674935365238753, 0.8594196530848439, 0.05805980864467484, 1.002157169778429], [-1.4049152983863993, 0.46719634154421597, 2.7787859008408984, 1.4409016354068573], [-1.5145898823674702, -1.3738358626710383, 1.4796287898528264, -0.3721319781174794]]], [[[1.4344258009787194, 0.6890273674567066, -0.12837789056078408, 0.8648175420625578], [-0.5987919421309226, 1.414376284166954, 0.12190407414562963, -0.1735475059739461], [1.017755283146379, 0.5600578849849129, 1.1904348079825622, -0.012817145825038666], [1.618107861034674, 0.008758685789787807, -0.18717491457684843, 0.23743960612113665], [-0.030966391875792837, -0.3490312379934012, -0.6483717290435569, 0.5292556770488435], [1.624098119006557, 0.5377844123347728, -0.4099457491208781, 0.3637541413380331]], [[2.83145926206279, -1.3679978599444067, -1.0529226240628695, 0.08011123813140524], [0.6931038819798587, -0.16071411370079602, -0.14309586519198847, -1.280603244081284], [1.342657248874597, -1.0089502661959102, 0.4864916422287545, 0.2354696413301572], [1.3270903127925884, -0.945645172145866, 1.2061842466436232, 0.19957878989135275], [-0.31445415196078325, 0.5120359155680366, 0.45171177675282004, 2.08433535642387], [-1.2060113907107002, 0.7470851738270198, -0.602609120734828, -0.6663622238943726]], [[-0.4643039819176934, 0.2716704512921767, -1.4090093831215509, -0.4101850093342803], [0.7683235838852829, 0.15437203400054636, 0.7040624763137266, 0.9961123451244088], [-0.048468965191317485, 0.6105445741179829, 0.11456710397256119, 1.2200445269154936], [0.8323754751343555, 0.26509455307388513, 0.5911160326104743, -1.2258481460456763], [0.18263492576528031, 0.21573624369474736, -0.4396102173338019, -1.0019622663057768], [-1.282214080522075, -3.4129767609481863, -0.5268297404563929, -1.5207524242651875]]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[[0.3201636658576962, 0.4285535071470855, 0.170328848207667, 0.8300293843398882, 0.7591947574186366], [-1.3251544023279311, -0.3256736932532758, -0.45468186085856077, 1.0825667225434652, -0.43680804950848917]], [[1.0607962575094456, 0.1468804461047493, 2.4004672997509955, 0.0057457171018046715, 0.059517304327636404], [-0.05677680127159672, 0.20182116062343144, 2.4828778224096033, -0.44958378012297495, -0.8182248601494634]], [[-0.14498426310628612, 0.7236251901268648, -0.9369281914883664, 0.8460574811881335, 0.8127056418772858], [-0.11686344043871084, -0.2049449388278705, -0.7416505201954731, -0.22885536572118775, -0.7548636763048272]]], [[[1.2864465329004582, 1.6693135638427257, -0.24161920694046624, 0.0004023177203800074, -1.8577793296682643], [-0.48929231143834323, 0.3772006019479164, 0.31029345829171423, 0.15976007028883385, 0.8809989686656728]], [[0.1265142479484446, -0.3527799109058192, 1.6342994963392032, -1.882397842676666, -1.1231029236824022], [-0.7060480214154771, -0.6988997841657503, -0.7454692984081157, -0.19863399668250917, -0.6876281000562338]], [[-0.034576441719339726, 0.6096417555607107, 0.0467729008432103, 0.14978359838658598, -0.9177576563090524], [-1.1184498209470457, -0.2779383229259432, 0.9361548429945662, 0.3869842429856282, 1.0952114291560793]]], [[[1.0011459104081328, 1.1301701753987994, -0.7932808239868254, 1.2445451003634107, 0.7653850634764275], [2.0772807021193334, 0.5977101132015179, 0.3354604334544054, -0.11743286337696804, 0.17548744807282013]], [[-1.632543838515133, -0.23683799547810405, -0.026065851441439677, 0.9965258559492283, -0.6405432368331196], [0.1437100921437122, 1.8275507855535846, -0.5140311166847379, 0.8498762932426861, -0.5180672939794508]], [[-0.08717534649858108, 0.4699875643988187, 0.889212343207782, -0.4767798106204294, -1.0020981000920175], [1.5188613500782844, 0.19799013117097164, 0.8002609201548928, 0.9185786652024484, 0.764309322764115]]], [[[-0.2461573096006576, -0.12190839480602847, -1.5274707520789663, 0.19133456194533924, -0.4519769731628085], [-0.009605283039135331, 1.8637509690002445, 0.6937499610071085, 0.06547140969748098, -1.3327446520965336]], [[0.7337110848877022, 1.5181533406406889, -0.8004674414728532, 0.4325746657660275, -0.4980714708783503], [-0.13662794882690169, -0.34792758295123033, -1.5764729083990014, -0.4930711463501457, -1.5852548515674725]], [[1.378226489746932, 1.0388893794857443, 0.05506785730697936, 0.28677728423391263, 0.06872920509825783], [0.2076172022635756, 0.20490533056392526, -0.22359072655165801, -0.151876092164102, -1.310432541714429]]], [[[-0.973034813459761, -0.5838696467040394, 0.21642760633777386, -0.3533463865662909, 1.885884123525062], [-0.17374433802016848, -0.3378542634313927, 0.4379376672840757, 1.0245460214839808, 1.83338677384235]], [[0.9708804863318135, -0.5633888760141369, -0.15201694197493976, 1.0467701238533533, 0.26900050971301687], [0.8014563146249944, -0.05585052821465467, 1.5882296492636734, 1.4376687432158106, 1.1231520774598631]], [[0.17433067237686975, -1.5570294960397313, -1.2990767738049098, 0.3423963323890769, 0.6638202710179099], [-2.826214767181137, -0.4391079838410445, 1.1522000287270784, 0.9599878940312865, -0.45984346218538713]]], [[[0.6500676210788175, 0.057154642367463176, -0.1855832321915171, 1.1234820274846915, 1.1430304788183245], [-0.48348756623872835, -0.8938437402264159, -1.2477530214808386, -1.35883361617638, 1.3349007862572024]], [[1.8648262682937249, 0.2404898535256428, 1.0629225241679479, -0.6329414334914988, -0.03174551523136807], [0.9313143492018942, 0.3449795625311947, 0.0025368695089940346, -1.5862555068705482, -1.380786343310924]], [[-0.3663721132847401, -0.5022204260811416, -0.03396618441613755, -0.584236769950118, -0.17405374743331445], [0.09766132158695162, 0.5410217915834006, 0.10228527926712694, -0.49703361121448614, -0.23362647261355213]]], [[[0.9192822405881155, -0.16407977336572885, -0.906081424599327, -0.9460440972615314, 0.31115854057116765], [0.5796923222946873, 1.2263473371549038, -0.2796524953816042, 1.3735371013284778, -0.3059794246347511]], [[-0.1946568611111578, -0.5217711251011656, 0.5411524015510516, 1.05014965429285, 0.004995407943487147], [-0.45856037142270734, -1.4729292603740158, 0.17212933322036997, -0.3063606825351341, -0.16902884490035067]], [[-0.5875093202919337, -0.8255769037857372, -0.39263432916798796, 0.4595503116041143, 0.954648765318467], [0.011284542702358638, 0.3360566958313506, -0.17004065785594555, -0.09982952431634311, -1.3070999199002638]]], [[[0.8779817801951034, 0.599260675421295, 1.4344312189101325, -0.5745570067887912, 0.14465259830432423], [-0.8759451750227435, 0.2539585442513139, -0.09897677055472931, 0.6767484390020514, -0.12040253515882449]], [[0.6449835298910125, -0.45464839925770023, 0.5981536817096281, -0.2716369809051092, -0.401935700655738], [0.10711113768361029, 0.3789922537218471, -1.3443832772445252, -1.4471773073208933, 0.20568685533172576]], [[-0.03210219655069302, -1.9469980184274847, -1.627336516248008, -0.7208100999026168, 0.333402207520583], [0.22141604383439378, 0.9998304445276331, 1.1592841922612263, 2.320032561235013, -1.3723599051638742]]]], "dtype": "float64"}, {"__nd__": [[[[-0.013785125437243632, 1.0572185971246792, -1.1914668744218657, -1.340640938265793], [0.2168166365317877, 1.0876883397547206, 1.2406552390816776, -1.6978177175038422], [-1.1518146074499096, -0.06735824086173853, -0.4867784758753965, 0.33945675195516584], [0.15675535674547536, 1.7359110240682145, -0.03344245520836803, 0.20497274176700866], [0.1019629089537021, 1.1016439719371758, -1.0495768502805989, 0.4038074180437287], [-0.7196693124627012, -1.3453354220650022, 1.1762868527500996, 0.5426834132063217], [1.390468251004086, 0.28276550720549054, 0.6671103167123836, 0.1693731609500687], [0.591748375812899, -0.3609093744289301, 1.7016025243391555, -0.8201969987485483]], [[1.7057604227119392, -0.1485859620526797, 0.892176926199896, 0.1808047362323036], [0.48228553424647513, 1.4990305072300323, 0.18873841932208604, -0.8698955250802021], [0.28480095562091245, 0.6849272083111604, -2.3436743704044205, 1.005519988231611], [1.1186067883318345, -1.6757684618477942, -0.09786126884998053, 0.8562725474733723], [-1.079271150493512, -0.3587975815314989, -0.2795021018962309, 0.6004593479606496], [1.1436135261476181, 0.022949778359614842, 0.11536831768857238, 0.9619436868261118], [0.5853611550534336, -0.05394911226220121, 1.3585437484020153, -1.3352926420597546], [0.20206536712334885, -0.3604046564861119, -0.4424982088183314, 0.2463545422981459]], [[0.08954466487739011, 2.46690278881854, 0.6843624219290668, 0.3969602654258929], [1.6746404517200135, 1.0493733221474362, -1.78126481722136, 0.17044976606605902], [0.7818632472974255, -0.5458417751114946, 0.3398814327604031, -0.40159310919948044], [0.41614665737346584, -1.5575698694674582, -1.874250860073316, -0.4973315808722033], [0.26492152714025896, 1.353139700388159, -0.23238167870158793, -0.6981062177955457], [0.9420227061786725, 0.07280247998546958, 1.8154182580065017, -0.050739112861102734], [-1.8845176486335493, -0.7511929750824291, -0.9051104687773869, -1.3399129207571248], [-0.7324886899933402, -0.648721184746771, -0.25051264651099836, -0.5116637290595857]]], [[[0.4315355666093416, 0.43589855951946865, 0.20217981812289446, 0.22221021497090085], [-1.834242062548122, -1.62150942471045, -0.8437504964266122, 0.2767344529418739], [-1.4215724115758346, -0.3913985875305514, 0.2697241076159053, 1.986707598419474], [0.28176867203175826, -0.5059260987027678, 1.1024009730351338, 0.22433927379257237], [-0.27569182256266134, 1.1198322753344498, -0.8906886071465294, 0.20543389798570286], [0.0937499619126291, 0.9072457236618531, -1.124213183944451, -1.2728910656074317], [-0.3783512423169322, -0.26639162525572263, 0.04562491338417157, -0.36877099437851346], [0.42910755917718413, 0.6845958868011797, 0.22120583596090126, -1.046443423606682]], [[1.1411778598317965, -1.5240790653030833, -0.20895319871138945, 0.8774041448217954], [-0.6181960721371497, 1.2746925814356092, 1.8907825930723234, 1.555187374087123], [0.42311325745906214, 0.9650139060808004, -0.5183897448315553, 0.38831854270175603], [0.4431072922419084, 0.08349949179373894, 0.14439734215986738, -0.21514803838443494], [-1.957853040525454, 0.38814400316153996, 0.30221122986793525, 2.0293054413629745], [0.6977156422983128, 0.05810807455394167, 0.8594762130279601, 0.6098951791404865], [-0.025558190380767246, -0.38735041735321674, -1.4679990982848865, 0.1134035293850709], [1.6074281555234813, -1.8636541397733377, -0.9536433443315753, 0.7432785985788312]], [[0.18359946403624242, 0.40937801445581334, -1.8577737406020323, -0.5945903747997203], [0.4945397663266917, 0.5778987880238712, 0.2858124878863606, -0.8860845221544852], [0.5290326274918391, -1.1968166283212809, -0.9440393605578705, -1.1677287703379882], [1.4259712220770964, -0.41940295640102204, 0.04600414498638384, -2.1984239922083333], [1.0813047511732246, 0.45218611680841353, -1.6580230816668347, -1.6098221547946951], [1.6429228631778157, -0.9511963888680472, 1.8240394312764485, -0.47790163397637253], [-2.485836227836113, 1.0933413869889934, 0.4894861752078354, 0.6681704173170244], [-0.5189090151079069, -0.2253305003519543, 0.9421473707408607, -0.8329755713839344]]]], "dtype": "float64"}, "dim head batch seq", "batch head dim seq"], "e": {"__nd__": [[[[0.11084550642623302, -0.26885409668847715, 1.6364740330248453, -1.0353952732500735], [-0.3319974865841715, 0.35292961550494983, 0.9208864447533267, -1.334355417578605], [0.12873203712228576, -1.048873150164629, 0.4686159566097326, -0.6165259715202203], [-1.3841079818053432, -0.2950750266429419, -0.536358859941323, 0.0446395786773236], [-0.5224650377764557, -0.5227658081103955, -1.3251206227505403, 1.2796621785512061]], [[1.1764297915457866, -1.0052159392157216, 1.4561344278388988, 0.6590569747402097], [0.7767393741754948, -1.0101989998491547, 0.052505661416023544, 0.6622069364522412], [2.049642711851221, 1.1493663730411108, 1.140257479987913, -0.47493863476025117], [-0.5035027200788326, -1.1362614474780284, -0.5468557808244842, 0.5520628607886117], [-0.5602512185088502, -0.4416367256801874, 0.5292297010846029, -0.020337551200677997]], [[0.44768300305717756, -0.6440563965392085, -0.9956878360329391, -0.004719283647767085], [1.4078137024744077, 0.10251572798666146, -0.5985799854187888, 1.025520887725265], [0.8019903365951104, -1.1465889084685474, 0.16891615925382578, 0.5370202001448986], [-0.25616469329070496, 0.9196139901958638, -0.6232820900011999, -0.016298988702326246], [-1.5027472142485736, 0.506957158229128, 0.10738131879194987, -0.4843281939927345]]], [[[-1.1394004633368073, -0.6999388956327874, 0.43331658791865285, 1.7598173079787973], [-0.5312589518928766, -1.1570260558978205, 1.1489187048698337, 0.7028530249666299], [-0.43174046526627513, -0.6429325105010105, 0.5231018949638324, 0.9517138702377642], [-0.09885941234812981, 0.08400439492102738, 0.3365871371846709, 0.2798642382650418], [-0.9707367539605148, 0.7954543361756042, -1.9190265327816256, -0.3137142446134076]], [[-0.12851636458348498, -0.12126357532932214, 0.0694116704667675, 0.409896437952402], [0.8055203642395358, 0.1409559744651967, -0.09927908962213888, 0.030551058774246128], [-1.0235150430768238, -0.8155160936761817, -0.1336283329017305, 1.2033292677805485], [-2.294226590299201, 1.5883493420598125, 0.03802888285840504, 0.2023553536761996], [-1.5053264828524984, -0.07917157611904643, -0.6664028548190768, -0.02708828140023793]], [[-0.8886516231864674, -1.4168616826265934, 1.2555822345522683, 1.1160318299602912], [-0.2539582521514924, -0.4025064155674112, 1.041284768762193, -0.16740507813131178], [0.5891563331079802, -0.22981774368960195, 0.05826624967360594, -1.3486172190845889], [-0.11155523525064716, -0.2230128674284318, -0.24739492407197403, -1.503555948786091], [0.7138397039729889, -0.4054635587384687, -0.23437684818515694, 0.9151583066481402]]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[[-1.9752289663311229, 0.8923466668544127, -0.6322474633461733, 1.257908403786471, -0.4956297469371905, -0.4407538480844032, -0.26612456691114955, -2.3663153516184674, 1.1586400685161409, -0.6432483247451606, -0.2107228449314508, 1.4466402801668148, 0.0999643327393738, 0.8790738118131123, 0.3202430195439585, 0.7567090193879319]], [[0.2622949430386147, 1.1387935191939222, -0.22413885288989877, 0.44651401996199686, -0.7214216176939628, 1.1713738440963957, -2.193752251993028, 0.17275453860774148, 0.3479059579266027, 0.32904184733365754, -0.019790936820598185, 1.32605513048693, 2.1014289506844084, -0.6073958218912924, -1.920087721324904, -1.0991789246794883]], [[-0.8243827790348474, -0.14810227413184754, -0.06763699215992872, 1.065465401403132, -0.439303310544801, -0.049629381137446964, 0.09713341415036728, -0.8906799710424794, -0.3303353297797456, 2.7425912649269066, -0.1051327237713318, -0.9753181988520102, -2.1901963798312636, 1.0241238807574213, -0.3356299385339455, -0.6474533088072583]], [[2.4308434360069775, 1.4182775237933014, -1.7253302604412548, -0.361422814281478, 0.983833747174225, 0.8207517923757125, -1.1270998714750489, 1.7620943182751083, 1.8358071520926498, -1.1227653073530883, -1.7803300412008762, -1.405688705221436, 0.65078180888151, 1.0621857267518904, 0.1674170511803008, -0.6129647968215749]]]], "dtype": "float64"}, {"__nd__": [[[[-0.11115107669435012, 0.545254858141902, -1.2326768414310398, -1.7168357499602045, -0.10561407115888492, 1.2133081983683347, 0.16984093343350606, 0.17263193767384075, -0.6962419493344834, 0.8977471367362945, -1.730201166124734, -0.08843024136350532, -0.6173286516794052, -0.1786744956595217, -0.9521907806099962, 0.7226251105474092], [-0.1045492421145289, -0.6244663105410415, 0.48045768618733453, -0.08159482319527207, -0.5501055677643735, -0.6215181178654333, 0.7462332611456225, -0.971940919235464, -0.2556901811830783, -0.30248296900544114, 0.2788255079485994, -0.22845868993732152, 2.121237474996319, -1.7823542231928142, 0.556312704028111, -0.06899503100453341], [1.0719588666913644, -1.6615168315226014, 1.1792610468154816, -1.4204143932246844, 0.6526589102116326, 0.4729805191233228, -0.007830379022175043, -0.7760050748041185, -0.8554891089622159, 0.05861421677340844, -0.7905116201121387, -1.9903665923814138, -1.8470695045967587, 0.760685603971935, 0.7855006800825647, 0.4787225607860483], [0.012259075653948777, 0.34625949206951206, 0.462128747511661, 0.2198761974886892, -0.8225317496665419, -0.7628558767505351, -1.1806306014359411, 0.12368922481839503, -1.0183985243208995, -0.1852106099497462, 1.2895961327685581, -0.4007664976345694, 0.9477011592431388, 1.5366466775155243, -1.8462075204294512, -1.4550038404841], [0.4535783510945984, -0.7275595058027274, 0.5174764476641356, 0.6908771675836225, -0.529038222168405, 0.8123423742916683, 1.7308965060403108, 0.9444819535275545, -0.8852678674487, -0.15462060260198104, -0.5446020244506633, 1.5613423783462355, -0.1986041954711762, -0.6944220961465117, 0.04367179292309263, 0.512468588054019], [-0.3763312929221769, 0.10705791647150001, 0.0024917888181681842, 0.7415816801593685, 0.977899922915871, 0.27968519992073515, -0.46035792854551966, 0.41395663783109893, -0.8825797169711659, 1.0847476247002674, 0.3603660068820682, 0.25744332652755664, -2.410298228197553, 2.3841859545994804, 1.4711592733235606, 1.2884744246430653], [0.1470593693472577, 0.6938448306349638, 2.881280480218307, -0.6559854506701525, 0.7759380468884297, 0.6342947135884008, 0.1764724502104331, 0.3914646628276239, -0.9390908907488439, 0.6915391283864756, -1.2631766282715235, 1.8821665981703952, 0.8521352630498372, -0.7967486176422764, -2.0363402272224307, 0.1634392410024217], [0.5680439618562257, -0.932784273942568, 1.7414985569027914, 0.3390537234134902, -0.39467482013214694, -1.3252567564302977, -0.04735439012433817, -0.12439973735030146, 0.72606419085965, 1.1257862763080106, 0.10802280943353589, 0.5389138081050272, 1.5030630262139961, 0.69677345489485, -0.17905554984171054, 0.6983010617032985], [-0.44238339738888083, 0.5312957168648914, 0.13035548612477726, -1.0187915497039697, 0.6830295921507366, 0.16322801708677048, 0.8106618983107764, -1.248614810217789, 1.1861758588428661, 1.228812012182094, -0.27415849201463965, -0.022449886189613556, 0.12253140402148537, 0.052425841776362676, 0.9073825586472529, -0.6430671844955037]], [[-1.0808987929251592, -0.17601643541176454, 1.9629716106012614, -0.3276088376484004, -0.9405286677684174, -0.8623668726452088, 0.22092879317545025, 1.2281937349924676, -0.6862431171352846, 0.22234377860987703, 0.2604827574978832, 0.5109481776803829, 0.1889545954697707, 1.094168858536071, 0.7297550115357907, -0.573339511502536], [2.467246400590262, -1.3218557604206154, -1.2772600633178357, 1.075500393048241, 0.2260152558475382, -0.20893940265557712, 0.23949386164930822, 0.7244843782012973, -0.38553954424869197, -0.5983303080304652, -1.0125545433537233, -0.8265764230720232, -0.7659786568229704, 1.1788495919599633, 0.31160625629532607, 0.5756569780118106], [-0.5320654969001571, 1.0790150520019992, -1.530859530658099, 0.5036996352018903, 0.7234464210268415, -0.5512312602485658, -0.9989850281663712, 0.5466574903276575, -0.684355625581703, -0.8702754212428183, 0.002018636353295629, -1.0957165211718947, -0.11396929369874481, 2.049099925915093, -0.4149257942839988, 0.9298010050668453], [-1.0312168237865709, 0.10533149780958398, -0.24947345008514368, -0.1456363774327925, 0.778255903674282, 2.1698338318657964, -0.10630506528992473, 1.3247079550450271, 0.37041557013123794, 0.756495271107219, -0.4486969354407436, -0.6872261648937884, -0.18156942627104783, -0.7536617148527954, 0.009123672628446803, -1.037537227525783], [0.023145682776247976, 0.19463421346064694, 1.833901634970986, 1.7419975764219133, 0.5283176292541396, -0.48369688761546414, 0.6267964932087111, 0.8358477573655989, -0.17668025360390052, -0.8293713415874506, -1.7712765667292065, -0.01900007581263784, 1.9416769432809233, -0.1866835638930086, 0.40568642115121056, 0.38567680397421855], [0.5051284530946324, 0.7945406790911973, -1.9836710882950375, -0.12875913582543813, 0.39883760477936037, -0.06705044788563704, 1.238708631183721, -0.3480604839268075, 0.94694933314951, 1.7920413979376701, -0.225408410929155, 0.057759689854807485, -0.2623107580602779, 0.7974582055588898, -2.1079932791879634, -1.4305156154425867], [-0.2600786165130346, 0.9074793656278695, 0.6277872523208214, -0.7818407250866336, 1.6447612466959614, 0.15085081300632777, -0.585057238734266, 0.572384282865618, 1.3474761079721327, -0.024584979351760152, -1.4759911564032755, -0.5078162999985072, 0.13803062220548454, 0.9861943007663295, -0.9580776343989692, 0.30879470181546226], [-1.1394537312464117, 0.12330487537458834, -1.286838023260862, -0.41652218432998855, -1.4775577777620972, -0.9329387406720938, -1.544848884159711, -1.9235821272837244, 1.2739397244795765, 1.2287387414728528, 0.5300590962423544, -0.3069829207249997, 1.5223149967766174, -0.2297925206308939, 0.48002253879242635, 1.191361558876349], [0.27901453444144625, 1.5368289784177398, -0.08661879376533692, -0.09483730679199344, 0.19175628895547037, -1.0959360568283227, -1.0146629677166434, 0.8395438306941063, -1.6665738675435875, 1.1950829718137463, 0.33906376414243583, 1.2519441704145653, 0.8313003134222399, 0.6680268528472278, 1.8131299255280966, -0.006619352547853983]], [[-0.03390422674948487, 1.1996454830879408, -0.5904881818980534, 1.2064158702872578, 0.761138038235218, -0.4319856629361248, -1.2055048646158915, -1.9559957408450315, 1.9460760910868768, -1.4253812154169319, -1.1413184976339208, 0.9324416781732131, 0.5389014833223098, 0.2745162801537509, 0.14609888714408487, -0.36570547211848337], [-0.8646954026399924, -0.6852209158459096, 0.6529103112395729, 1.1334610460907835, 1.2387706667579075, 0.014396779583904225, -2.200051522409038, 0.5596252520533068, -0.09936885394129398, -2.6409965431829794, 1.4376650413769236, 0.2722074182441439, -0.0033549124554330383, 0.2493941126507262, 0.1734665214329772, 0.29236547229250903], [0.17090200888274532, 0.0797055463175671, 0.6305605623944122, 0.40724401026369916, 2.4805535330603887, 0.20415771151723086, -0.4472391314039362, -0.5324724312922569, 0.45582912267271297, 0.19513754983014567, 0.3220834534110397, -0.12491129805826703, -0.10494505755139429, 0.8335395910660917, 1.4451780394262974, 0.42491819935847513], [-0.02818743877931534, -1.0845687784296523, -1.0856214859314706, 0.12048393678023446, -0.1753551510067229, -0.029498199803147756, -1.0472145841254168, -0.06253773154365819, -1.8932444072386205, -1.1398168646916886, -0.7421925465523216, 0.6393009872248205, -0.5662417435702584, -0.9038388876993864, 0.3102126186364812, -0.06312231121284918], [0.9578329360021163, 0.3540431369914342, 0.24079761488734586, -0.16631867822320587, 0.6837397536358178, -0.7159861631319479, -1.51558747359909, 0.918843181006387, -0.46504669736809845, 0.05570192484096737, 0.6328984648504472, 1.025900174912611, -2.9031770986108767, -1.0468684994612352, 1.2337498568239031, -1.7055178688042596], [0.4762640204533326, -0.3627007147570221, -0.8921003473140369, -0.1311357100544179, -1.6965107688639924, 0.40589581788929163, 1.1765063507154565, -1.4324692652197635, -0.6737122392175654, 1.2012774669126218, 0.36823460066437547, -0.3279290848169645, 1.4351848592627359, -1.6871209489354442, -0.2809986039526827, 0.08910385937046722], [-0.3242110354209239, 0.794981381491696, -1.6841487500593129, 0.041941562851711155, -0.7951081415674971, 1.1987022041105742, 1.0446423518539394, -0.7540777802874681, -1.1109820020707983, -0.052439053913775, -0.8694816529005828, -0.5787657299936958, 0.42370518876820945, -0.4685525448590709, 1.8138983104403004, 0.05952042714792125], [-0.5323242238402112, -1.130155475111004, -0.5311712846002009, 1.1020460768927398, 0.5418751503206436, 1.42530197137286, 1.306351275045417, 0.548682031430971, -0.7083648635131103, -0.16049906773410771, 0.0593281132308795, -1.6690553524266645, 0.32165477287610617, -1.051837571530199, -0.09585031453224119, -0.5272992347338971], [0.5736537873923023, 2.633124182048203, -1.2069086920966186, -0.02718657927201162, 2.0459627168303367, -0.9381708609247682, -0.23343836584960526, -0.11366590831417316, 0.3066661714446259, -1.6295677544234592, 0.3029867200518793, 0.6034652204499303, 2.554124348895932, -0.08022479597738962, -0.5807622151140671, 0.132237630904046]], [[-2.6241302063567864, 0.3390107586695493, 1.5652866685280724, -0.033104042013450866, -0.03400511070018024, -0.24262819173228103, 0.14173579275885118, -1.2457361175160275, 1.5561570533274256, -0.4559412114743311, -0.45000738564170256, 1.6717228111130935, 0.031383263378400694, -0.4107483649460389, 1.670682173540645, 0.19934625526628008], [0.9619471694139166, -0.5526202269171941, -1.7544792586000593, 0.19771326382339222, 1.4599751038732074, -0.4394121199439751, 0.3906843422063216, 0.7519165626696607, 0.2914081290894561, 0.10033553754856293, 0.8053041089350148, 0.24408949287004442, -0.5051175794881416, 1.092796201860871, 0.8403662705193835, 0.4125444934268037], [0.7595425296013048, -2.613989625276266, 0.30585155994589763, 0.07747472120928245, -1.9600838507190712, 2.1217827106599323, 0.3910723039505232, -0.40222343425446494, -0.33886683641278315, 0.3079967578018055, 0.024515388931280264, -1.5770111432522806, -0.3604246827397394, 0.4914362712644064, 0.6596852718067866, 0.913526168028074], [-1.0220931280243284, -0.5900863276185884, 0.42004520645025717, -0.5641858829280169, 0.26408806457479916, -0.7852447508828215, 0.640905623249587, 0.11563713807487899, 0.6402919285477774, 0.09382452814501326, -0.7299577838888034, -1.1644746476129855, 0.10823257324411059, -0.5488093941677714, -1.4282311889764632, -0.483622678892862], [0.863950402308464, 1.3144090056994802, 0.8382377978980102, 1.4408756446997086, -0.2963641106855874, -0.0728256438988777, -0.8781379106946264, -0.6975635133020428, 1.699877996375388, 0.5192322905573097, 0.7358409689034593, 2.7045630063910218, 2.023057304853769, -0.7820925331907388, 1.0720834395459993, -0.6707949749156548], [1.0838982800395982, -1.8001729885630262, -0.8540792557181873, -0.8314402188719684, 2.1068797392589156, -1.696012186084761, 1.209098796738431, 1.1745224765392945, 0.21123375192596835, 0.9305042022831841, 2.690395392542448, 0.34725517607339473, -0.18592857168528626, 1.4567855333389779, 0.8817878464933996, -1.0671946275346311], [0.2814823554127324, 0.37875786317881355, -1.0624080500848743, -0.8689409157521849, -0.5779282154935942, -1.12565196816772, 0.6099210835711482, 0.3916228739762889, -2.158439395047798, -0.3419791697876799, 0.24037520626694772, -1.2846588027214927, -0.20182561040711608, 0.3723554453125937, -1.6439969128003826, -0.2357605897959158], [-2.3244585111997336, 0.8270865091074265, -0.67880320898866, -1.4082980621701646, -1.8134986026793718, -1.618625787412987, 1.3463342262237525, -0.8091784191799666, -0.5387489319636927, -0.3628352131759902, -1.4947246863810781, 0.804060451554693, -2.5145754528110658, 0.9299538693048875, -0.7476893109801673, 0.06226577787981531], [0.31638235477806964, 0.8647771752653872, -1.2451944404966013, 0.63169529663546, -0.04253497565362507, -0.14972335430628944, -0.27905034161739967, 1.1402994126773978, 0.4842745463396784, 1.2972814898808864, 0.9357013919275518, -2.009154634199146, -0.10172494882345426, 1.027011072766268, -0.9157797945822365, -1.7093495669058933]]]], "dtype": "float64"}, "batch head seq dim", "batch head seq dim"], "e": {"__nd__": [[[[-0.683670447168221, 0.04275344644964922, -1.8674370916944474, -0.28110932681015244, -0.643998622713121, 0.5455870352698429, -0.9107957596007447, 0.14138207902497602, 0.6959671440393002]], [[-0.5515128401600776, -1.4658721776991939, -0.22782474814252732, 0.73928834887434, 0.27060681325487024, 0.9662748144119759, 0.4639092470153899, 1.1277953674386798, 0.5207941872561567]], [[-0.8912998294323651, -1.7227423318805277, 0.09136155152455891, -0.5941765143307713, 0.7374606176158964, 0.16518718157731618, 0.04873216732475748, 0.2882630705594551, -3.085436167423345]], [[-2.403139501490563, 1.4589617060158366, -0.6775104282843257, -0.2985347688504878, -0.022771743662649183, -0.11583747129368785, -0.04713387238396477, -2.178998350549217, 2.146185659889592]]]], "dtype": "float64"}}, {"a": [{"__nd__": [[[[-2333.91327743001, 2019.7602672884227], [-942.3668252427032, 169.15523301360926]], [[-1099.6614095493267, 426.1535091295774], [1680.6993549396566, -1483.9747175787138]]], [[[-71.35990475149464, -786.5275439953831], [-1921.9656825224783, -918.0206699604387]], [[22.351490029677407, 1253.98451544462], [1299.8330796765902, -21.15975347473603]]], [[[-1693.7263592704994, 671.8358614804459], [-8.554019076600584, -19.56520390313709]], [[-1172.8698066822822, -162.62624705008483], [2004.9469141989982, 1521.6397508612579]]]], "dtype": "float64"}, {"__nd__": [[[[1.301401992750284, 0.6831285076860435], [-0.4063545332881459, 0.001902143957527719]], [[0.15455844167019225, 0.39915325077888747], [-0.3562451937734825, -0.8836852215910307]]], [[[1.1446408693320411, -0.38013989588685176], [0.8623542379672526, -0.1768647339215874]], [[0.6954369768358598, 0.5290508427948168], [0.7601552232444927, 0.707010421669371]]], [[[-0.9656563774045915, -1.38418460520268], [-0.04374552188000326, 1.6943448865519983]], [[-0.8474162536814941, -0.9452128314400859], [2.330396625844131, -1.5767143693360588]]]], "dtype": "float64"}, "head seq batch dim", "head seq batch dim"], "e": {"__nd__": [[[[-1172.1027269006993, 314.9933568602934], [-806.0903737115149, 0.0980083056745164]], [[153.66069245700928, -329.32747136029707], [-318.97948525499714, 480.1011922572119]], [[498.9435984049425, 565.8716551259868], [960.03460324507, 811.4936694213085]]], [[[271.0034744815871, 131.68709284817535], [-484.92148128292126, 503.9022950450141]], [[-1057.1598305952257, -1492.0253132739997], [795.2539906278903, 688.0960305617664]], [[-23.17613409388534, 7.717703485487027], [1761.0318188177998, 1607.3458239897498]]]], "dtype": "float64"}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(attention_scores, json.loads(_TESTS), 'torch', *[1e-06, 1e-08])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")